# RAG 실험 시작 노트북

이 노트북은 **환경 세팅 확인**과 **기본 동작 테스트** 용도입니다.  
청킹·임베딩·DB 구성은 각자 `{이름}_exp01.py`에서 자유롭게 구현하세요.

---

**순서**
1. 환경 확인 (API 키, 패키지)
2. PDF 데이터 불러오기
3. LLM 출력 확인

## 1. 환경 확인

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"))
api_key = os.getenv("OPENAI_API_KEY", "")
print("API 키:", "OK" if api_key else "없음 — .env 파일 확인")

# PDF 존재 여부
data_dir = Path("../data")
pdfs = list(data_dir.rglob("*.pdf"))
print(f"PDF 파일: {len(pdfs)}개")
for p in pdfs[:5]:
    print(" ", p.relative_to(data_dir))
if len(pdfs) > 5:
    print(f"  ... 외 {len(pdfs)-5}개")

## 2. PDF 데이터 불러오기

PDF에서 텍스트를 추출하는 가장 기본적인 방법입니다.  
어떻게 나눌지(청킹)는 본인이 정하세요.

In [ ]:
import fitz  # PyMuPDF

# 첫 번째 PDF로 테스트
sample_pdf = pdfs[0]
doc = fitz.open(str(sample_pdf))

full_text = "".join(page.get_text() for page in doc)
print(f"파일: {sample_pdf.name}")
print(f"페이지 수: {len(doc)}")
print(f"텍스트 길이: {len(full_text):,}자")
print()
print("--- 앞 500자 미리보기 ---")
print(full_text[:500])

## 3. LLM 출력 확인

OpenAI 연결과 답변 형식을 확인합니다.  
실험에서 쓸 `SYSTEM_PROMPT`와 동일한 설정입니다.

In [ ]:
from openai import OpenAI

oai = OpenAI(api_key=api_key)

SYSTEM_PROMPT = (
    "너는 가전제품 사용법과 문제 해결을 도와주는 어시스턴트야. 반드시 한국어로만 답해. "
    "아래 검색된 문서 내용만 근거로 자연스럽고 친절하게 답변해. "
    "문서에 없는 내용을 지어내면 안 돼."
)

# 임시 컨텍스트로 LLM 동작 확인
test_context = full_text[:800]  # 앞 800자를 컨텍스트로
test_query = "이 제품 관련해서 간단히 설명해줘"

resp = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"[문서]\n{test_context}\n\n[질문]\n{test_query}"},
    ],
)
print(resp.choices[0].message.content)

---

세팅이 정상이면 `{이름}_exp01.py`를 열어서 `my_answer()` 구현을 시작하세요.